# LiveClip Studio AI — Colab Pipeline

Backend for the [LiveClip Studio AI](https://github.com/Sgsuprio/LiveClip-Studio-AI) web app.
Run the cells top to bottom on a GPU runtime (Runtime → Change runtime type → T4 GPU).

1. Upload `liveclip_config.json` (exported from the Studio's Share & Export panel) when asked, or edit the defaults in cell 3.
2. The pipeline downloads the source, transcribes it with faster-whisper large-v3, scores viral hooks, then renders vertical MP4 clips with ffmpeg + burned-in captions.

In [ ]:
# Install the pipeline dependencies (yt-dlp, faster-whisper, torch)
!pip install -q yt-dlp faster-whisper
!apt-get install -y -qq ffmpeg > /dev/null
print("deps installed")

In [ ]:
import json, os

# Defaults match the Studio UI. Uploading liveclip_config.json overrides them.
CONFIG = {
    "source_url": "https://www.youtube.com/watch?v=dQw4w9WgXcQ",
    "target_duration": "15-30s",
    "aspect_ratio": "9:16",
    "max_clips": 3,
    "dynamic_captions": True,
    "caption_style": "hormozi",
    "smart_face_tracking": True,
    "viral_hook_detection": True,
    "auto_silence_removal": False,
}

if os.path.exists("liveclip_config.json"):
    with open("liveclip_config.json") as f:
        exported = json.load(f)
    CONFIG["source_url"] = exported.get("source_url", CONFIG["source_url"])
    CONFIG.update(exported.get("clip_settings", {}))
    CONFIG.update(exported.get("ai_pipeline", {}))
    print("Loaded liveclip_config.json")

os.makedirs("/content/liveclip", exist_ok=True)
print(json.dumps(CONFIG, indent=2))

In [ ]:
# Stage 1 — Ingest the source stream with yt-dlp
!yt-dlp -f "bv*[height<=1080]+ba/b[height<=1080]" --merge-output-format mp4 -o "/content/liveclip/raw.%(ext)s" "$CONFIG[source_url]" 2>&1 | tail -5
!ls -lh /content/liveclip/
# Print the real source duration (ffprobe reads the file, so no hardcoded guess)
!ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 /content/liveclip/raw.mp4


In [ ]:
# Stage 2 — Transcribe with faster-whisper large-v3 (word-level timestamps)
from faster_whisper import WhisperModel

model = WhisperModel("large-v3", device="cuda", compute_type="float16")
segments, info = model.transcribe(
    "/content/liveclip/raw.mp4", beam_size=5, word_timestamps=True, vad_filter=True
)
words = []
for seg in segments:
    for w in seg.words:
        words.append({"start": round(w.start, 2), "end": round(w.end, 2), "word": w.word})
print(f"Detected language: {info.language} | words: {len(words)}")
with open("/content/liveclip/words.json", "w") as f:
    json.dump(words, f)

In [ ]:
# Stage 3 — Score hook windows and pick the top clips (starter heuristic:
# high word-density windows with question/exclamation openers score higher).
# Replace with your LLM hook scorer when ready.
import re

DUR = {"15-30s": (15, 30), "30-60s": (30, 60), "60-90s": (60, 90)}[CONFIG["target_duration"]]
WINDOW = DUR[1]  # score windows of the max target length

candidates = []
for i in range(len(words)):
    start = words[i]["start"]
    window_words = [w for w in words if start <= w["start"] < start + WINDOW]
    if len(window_words) < 20:
        continue
    text = " ".join(w["word"] for w in window_words)
    score = len(window_words)  # speech density
    if re.search(r"\?|!|secret|mistake|stop|why|how", text, re.I):
        score += 15
    candidates.append((score, start, start + WINDOW, text[:120]))

# De-duplicate overlapping windows, keep the top N
candidates.sort(reverse=True)
picked, used = [], []
for score, s, e, text in candidates:
    if all(abs(s - u) > WINDOW for u in used):
        picked.append((score, s, e, text))
        used.append(s)
    if len(picked) >= CONFIG["max_clips"]:
        break
picked.sort(key=lambda c: c[1])
for score, s, e, text in picked:
    print(f"{score:4d}  {s:8.1f}s - {e:8.1f}s  {text}...")
with open("/content/liveclip/picks.json", "w") as f:
    json.dump([{"score": s, "start": a, "end": b} for s, a, b, _ in picked], f)

In [ ]:
# Stage 4 — Render vertical clips with ffmpeg (center-crop 9:16) + .srt captions
import subprocess

CROP = {"9:16": "crop=ih*9/16:ih", "16:9": "crop=iw:iw*9/16", "1:1": "crop=min(iw\\,ih):min(iw\\,ih)"}[CONFIG["aspect_ratio"]]

for i, (score, s, e, _text) in enumerate(picked, 1):
    # Build an .srt for this clip from the word timestamps
    clip_words = [w for w in words if s <= w["start"] < e]
    srt_path = f"/content/liveclip/clip{i}.srt"
    with open(srt_path, "w") as f:
        for j in range(0, len(clip_words), 4):
            chunk = clip_words[j:j + 4]
            def ts(t):
                t -= s
                return f"{int(t // 3600):02d}:{int((t % 3600) // 60):02d}:{int(t % 60):02d},{int((t % 1) * 1000):03d}"
            f.write(f"{j // 4 + 1}\n{ts(chunk[0]['start'])} --> {ts(chunk[-1]['end'])}\n{' '.join(w['word'] for w in chunk).strip()}\n\n")
    out = f"/content/liveclip/clip{i}.mp4"
    vf = f"{CROP},scale=1080:1920,subtitles='{srt_path}':force_style='FontSize=28,PrimaryColour=&HFFFFFF,OutlineColour=&H000000,BorderStyle=1,Outline=2'" if CONFIG["dynamic_captions"] else f"{CROP},scale=1080:1920"
    cmd = ["ffmpeg", "-y", "-ss", str(s), "-to", str(e), "-i", "/content/liveclip/raw.mp4",
           "-vf", vf, "-c:v", "libx264", "-preset", "fast", "-c:a", "aac", out]
    subprocess.run(cmd, check=True, capture_output=True)
    print(f"clip{i}.mp4  ({e - s:.0f}s, score {score})")

!ls -lh /content/liveclip/*.mp4